# TCEM v0.5 Horizon — Enterprise Predictive & Preemptive Defense

This notebook runs the same TCEM v0.5 engine and web workbench inside one Colab VM.

**Default profile:** GLiNER ON · Qwen ON · Horizon ON · review gates ON.

Horizon scores **defensive forecast pressure**, not attack probability. Generated paths and actions remain candidates until analyst review.

In [ ]:
import importlib, os, pathlib, site, subprocess, sys
os.environ.setdefault('TCE_AI_GLINER', '1')
os.environ.setdefault('TCE_AI_QWEN', '1')
os.environ.setdefault('TCE_QWEN_MODEL', 'auto')
os.environ.setdefault('TCE_QWEN_4BIT', '1')
REPO = 'https://github.com/Ridd1kulusC0d3r/Cyber-Threat-Context-Modeling.git'
BRANCH = os.environ.get('TCE_BRANCH', 'main')
ROOT = pathlib.Path('/content/Cyber-Threat-Context-Modeling')
if ROOT.exists() and (ROOT / '.git').exists():
    subprocess.run(['git','-C',str(ROOT),'fetch','origin',BRANCH,'--depth','1'],check=True)
    subprocess.run(['git','-C',str(ROOT),'checkout','-B',BRANCH,'FETCH_HEAD'],check=True)
else:
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO,str(ROOT)],check=True)
os.chdir(ROOT)
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','-e','.[colab]'],check=True)
SRC = ROOT / 'src'
if str(SRC) not in sys.path: sys.path.insert(0,str(SRC))
site.addsitedir(str(SRC)); importlib.invalidate_caches()
import tce
print('TCE', tce.__version__, 'from', tce.__file__)
print('GLiNER:', os.environ['TCE_AI_GLINER'], '| Qwen:', os.environ['TCE_AI_QWEN'])
subprocess.run(['nvidia-smi'],check=False)


In [ ]:
from tce.io import load_case
from tce.validation import validate_entities
CASE_DIR = ROOT / 'examples/cases/enterprise-identity'
entities, errors = load_case(CASE_DIR)
errors += validate_entities(entities)
assert not errors, errors
print('Case entities:', len(entities), '| semantic validation: OK')


In [ ]:
from tce.conformance import evaluate_horizon_conformance
evaluate_horizon_conformance(entities)


## Horizon assessment

Offline mode uses the enterprise Environment State and its local telemetry requirements. Sync ATT&CK later if you want Data Component enrichment.

In [ ]:
from tce.horizon import horizon_assessment
horizon = horizon_assessment(entities, max_hops=5, limit=25, attack_index=None, budget=8)
horizon['summary']


In [ ]:
print('TOP CANDIDATE PATHS')
for row in horizon['candidate_paths'][:10]:
    print(row['forecast']['band'], row['forecast']['proposed_score'], ' -> '.join(row['nodes']))

print('\nRISK-WEIGHTED CHOKE POINTS')
for row in horizon['choke_points'][:10]:
    print(row['node_id'], 'leverage=', row['defensive_leverage'], 'paths=', row['candidate_paths'])


In [ ]:
print('DETECTION BY ABSENCE')
for row in horizon['detection_by_absence'][:10]:
    missing = [*(row.get('missing_telemetry') or []), *(row.get('missing_attack_data_components') or [])]
    print(row['band'], row['path_id'], 'missing=', ', '.join(missing))

print('\nPREEMPTIVE ACTION PORTFOLIO')
for row in horizon['preemptive_actions']['selected']:
    print(row['name'], 'cost=', row['cost'], 'marginal reduction=', row.get('marginal_risk_reduction'))


## Optional ATT&CK observability enrichment

This uses TCE's existing ATT&CK STIX cache and Detection Strategy/Data Component parser. It does not need a second ATT&CK library.

In [ ]:
from tce.standards.attack import AttackIndex
# Uncomment to download/cache current ATT&CK data and enrich Detection by Absence.
# attack = AttackIndex.load(sync_if_missing=True)
# horizon_attack = horizon_assessment(entities, max_hops=5, limit=25, attack_index=attack, budget=8)
# horizon_attack['detection_by_absence'][:5]


## AI review layer

GLiNER and Qwen remain advisory. They can help analyze Evidence Packets, but they do not silently accept intelligence, forecasts, paths, or defensive actions.

In [ ]:
from tce.ai.config import AIConfig
from tce.ai.copilot import TCECopilot
config = AIConfig()
print(config)
copilot = TCECopilot(config)
print('AI configured. Models load lazily when analysis is requested.')


## Web Workbench — port 3000

The web UI includes Context, Horizon, Engineering, ATT&CK, AI and Graph views.

In [ ]:
import time, urllib.request
from IPython.display import HTML, display
PORT = 3000
os.environ['TCE_CASE_DIR'] = str(CASE_DIR.resolve())
if 'tce_ui_proc' in globals() and tce_ui_proc.poll() is None:
    tce_ui_proc.terminate()
    try: tce_ui_proc.wait(timeout=3)
    except subprocess.TimeoutExpired: tce_ui_proc.kill()
tce_ui_proc = subprocess.Popen([sys.executable,'-m','uvicorn','tce.webapp:app','--host','0.0.0.0','--port',str(PORT),'--log-level','warning'],cwd=str(ROOT),env=os.environ.copy())
for _ in range(30):
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/health',timeout=1) as response:
            if response.status == 200: break
    except Exception: time.sleep(1)
else: raise RuntimeError('TCE UI did not become ready')
from google.colab import output
proxy_url = output.eval_js(f'google.colab.kernel.proxyPort({PORT})')
display(HTML(f'<a href="{proxy_url}" target="_blank" style="display:inline-block;padding:12px 18px;background:#23d1b5;color:#03100e;border-radius:10px;font-weight:700;text-decoration:none">Open TCE v0.5 Horizon Workbench</a>'))
print(proxy_url)
